### Realizamos la conexion hacia nuestro ADLS por medio de service principal

In [0]:
%run ../config/Access_ADLS_Service_Principal

In [0]:
%run ../includes/configuration

In [0]:
%run ../includes/common_functions

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

### Leemos el archivo person.json de nuestro contenedor bronze

##### El archivo Json en el campo "personName" tiene otro json anidado donde tiene el apellido y nombre separado, por ende tenemos que crear un esquema para la estructura de ambos json

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import concat, col, lit, current_timestamp


In [0]:
name_schema = StructType(fields=[
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])

In [0]:
persons_schema = StructType(fields=[
    StructField("personId", IntegerType(), False),
    StructField("personName", name_schema)
])

In [0]:
df = spark.read \
    .schema(persons_schema) \
    .json(f"{bronze_folder_path}/person.json")

###### Concatenamos el nombre y apellido en una nueva columna

In [0]:
df_concat = df.withColumn("name", concat(col("personName.forename"), lit(" "), col("personName.surname")))

###### Eliminamos las columnas que no nos interesan

In [0]:
#Eliminamos la columna que contenia el json anidado
df_drop = df_concat.drop(col("personName"))

###### Renombramos las columas y adicionamos columnas de control

In [0]:
df_renamed = df_drop.withColumnRenamed("personId", "person_id")

###### Adiccionamos dos nuevas columnas, una para guardar la fecha de ingestion y la otra para guardar el ambiente

In [0]:
df_add = add_columnas_control(df_renamed,v_environment)

###### Escribimos los datos en nuestro contenedor silver del data lake

In [0]:
df_add.write.mode("overwrite").parquet(f"{silver_folder_path}/person")